In [4]:
import requests
import pandas as pd
import json
import openai
from openai import OpenAI
import time
import re
import os

# check disease name 2 underwriting

In [5]:
df_dise_2_conc = pd.read_csv("./utils/disease_2_conclusion_v4.csv",keep_default_na=False)
df_dise_2_keyword = pd.read_excel("./中再疾病函数对照表.xlsx",keep_default_na=False)
# df_dise_2_keyword_kl = df_dise_2_keyword[df_dise_2_keyword["KL_match_disease"].apply(lambda x:x!="")]
df_dise_2_keyword_kl = df_dise_2_keyword

In [6]:
#构建疾病及其相关关键词map, df来自中再疾病函数对照表
disease_n_2_list = {}
for idx,row in df_dise_2_keyword_kl.iterrows():
    disease_n = row["disease_n"].strip()
    disease_list = eval(row["disease_list"])
    # print(type(disease_list))
    try:
        if disease_n  not in disease_n_2_list and disease_n!="":
            disease_n_2_list[disease_n] = []
        disease_n_2_list[disease_n].extend(disease_list)
    except Exception as es:
        print(es)




''
''
''
''
''


In [7]:
#大模型
client = OpenAI(
    api_key="token-vulcan",  # 输入你的 API Key
    base_url="http://219.147.99.170:50020/v1"
)


def qa_base(messages):
    completion = client.chat.completions.create(
        model="Qwen2-5-14B",
        messages=messages,
        logprobs=False,
        # stream=True  # 开启流式输出
    )
    return completion.choices[0].message.content
    # if compile.status_code == 200:
    #     result = completion[0].choices[0].delta.content
    #     return result
    # else:
    #     print(f"请求失败，状态码: {completion.status_code}")
    #     return ""
    

# 编辑距离计算疾病相似度

In [8]:
def edit_distance(str1, str2):
    """计算两个字符串的编辑距离

    Args:
        str1: 字符串1
        str2: 字符串2

    Returns:
        int: 编辑距离
    """

    m = len(str1)
    n = len(str2)

    # 初始化二维数组dp，dp[i][j]表示str1[:i]和str2[:j]的编辑距离
    dp = [[i+j for j in range(n+1)] for i in range(m+1)]
    for i in range(1, m+1):
        dp[i][0] = i
    for j in range(1, n+1):
        dp[0][j] = j

    for i in range(1, m+1):
        for j in range(1, n+1):
            if str1[i-1] == str2[j-1]:
                cost = 0
            else:
                cost = 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)

    return dp[m][n]

def normalized_similarity(str1, str2):
    distance = edit_distance(str1, str2)
    max_len = max(len(str1), len(str2))
    similarity = 1 - distance / max_len
    return similarity

# 示例用法
str1 = "kitten"
# str2 = "sitting"
str2 = "kitte"

distance = edit_distance(str2, str1)
print("编辑距离:", distance)

distance_nor = normalized_similarity(str1,str2)
print(round(distance_nor,4))


编辑距离: 1
0.8333


In [9]:
# 计算疾病相似度，并返回top_n最相近的疾病
def disease_similarity(input_dise,disease_n_2_list,top_n=3):
    disease_similarity_score = {}
    for k_dise,v_diseKey in disease_n_2_list.items():
        disease_similarity_score[k_dise] = max([normalized_similarity(input_dise,keyw) for keyw in v_diseKey])

    sorted_dict = sorted(disease_similarity_score.items(), key=lambda x: x[1], reverse=True)
    result =sorted_dict[:top_n]
    return result
    

# 读取 中再疾病函数对照表的disease_n 到 disease_2_concluson中的disease_name的 map
# 读取 中再疾病函数对照表的disease_n 到 key word 的map

In [29]:
#测试样本
df = pd.read_csv("./results_edit_distance/核保结论_规则引擎_vs_RAG_editDistanceSearch_20241204.csv",encoding="utf-8",keep_default_na=False)

In [30]:
df.columns

Index(['姓名', 'disease_name', '编号（身份证号）', '性别\n（1:男，2:女，0:未知）', '医院', '日期',
       '临床诊断（化验项、疾病等以下划线拼接）', '账单金额（数字）', '年龄', '编号（案件编号）', '图片名/文件名',
       'attach id', '图片分类/票据类别', '影像报告内容image_report',
       'data_source：类别 如昆仑体检告知', '线上页面与结果', 'label_result', 'RAG核保结论',
       'recall_disease', 'recall_conclusions'],
      dtype='object')

In [31]:
df_test = df[df["label_result"]!="材料不足"]

In [36]:
for idx,row in df_test.iterrows():
    label = row["label_result"]
    predict = eval(row["RAG核保结论"])
    if label != predict["重疾险"]:
        print(idx)
    



1
13


In [41]:
print(df_test.shape)

(17, 20)


In [47]:
print(f"""
测试数据总量：17
16个正确 
1个错误 
正确率{round(16/17*100,2)}%
""")


测试数据总量：17
16个正确 
1个错误 
正确率94.12%



In [50]:
14/15

0.9333333333333333

In [13]:
map_key_2_dise = {
    "胆囊疾病":"胆结石",
    "肺结节病":"肺结节",
    "脂肪肝":"脂肪肝*",
    "乳腺结节":"乳腺结节、囊肿、占位、异常回声",
    "肾结石":"泌尿系结石（无高血压和肾功能损害）",
    "心率不齐":"心率失常",
    "肺动脉瓣关闭不全":"肺动脉瓣疾病",
    "胆管结石":"肝内胆管结石",
    "卵巢囊肿":"附件/多囊卵巢综合征"
}

#疾病相关备注
map_disease_comment = {
    "乳腺结节、囊肿、占位、异常回声":["乳腺影像检查包括乳腺超声、钼靶、MR检查等；超声与钼靶针对不同检查方向，核磁具备更高的检查准确度"],
    "子宫出血":["对子宫疾病及其并发症除外"],
    "子宫肌瘤":["手术指征包括：持续症状（腰痛腰酸）、月经过多、贫血、直径≥6cm、内膜增厚、临近更年期肌瘤增大、短期内迅速增大等","对子宫疾病及其并发症除外"],
    "子宫内膜异位症":["对子宫内膜异位症及其并发症除外"],
    "子宫脱垂":["对子宫脱垂及其并发症除外"],
    "盆腔积液":["对盆腔炎、盆腔积液除外"],
    "宫颈上皮内瘤变":["对宫颈原位癌、宫颈恶性肿瘤及其转移癌出险，我司不承担保险责任","对宫颈疾病及其并发症除外","手术切除包括宫颈锥切（LEEP）、子宫全切","TCT-未见上皮内瘤变或恶性细胞（NILM）/炎症反应性细胞改变"],
    "宫颈炎":["对宫颈疾病及其并发症除外",],
    "HPV感染":["如有TCT检查异常、或宫颈既往病史，请参考【宫颈上皮内瘤变】","TCT-未见上皮内瘤变或恶性细胞（NILM）/炎症反应性细胞改变"]
}


In [14]:
all_dise_conc_keys = set(df_dise_2_conc["disease_name"].tolist())

In [ ]:
df_new = df[df["label"]]

In [40]:
gender_map = {2:"女性",1:"男"}
results_underwriting = []
recall_query = []
recall_keys = []
recall_time_use = []
for idx,row in df_test.iterrows():
    label = row["label_result"]
    predict = eval(row["RAG核保结论"])
    if label == predict["重疾险"]:
        continue
    gender = gender_map.get(row["性别\n（1:男，2:女，0:未知）"],"未知")
    age = row["年龄"]
    diagnose = row["临床诊断（化验项、疾病等以下划线拼接）"]
    image_report = row["影像报告内容image_report"].replace('\\n',"")
    image_report = image_report.replace("\\",'').strip("\"").replace("\'","\"")
    basic_info = f"年龄:{age},性别:{gender},临床诊断:{diagnose},影像报告:{image_report}"
    # print("input:",diagnose)
    # print("disease name:",row["disease_name"])
    print("病人基本信息",basic_info)
    # print(image_report)
    # if image_report != "":
    #     image_report = json.loads(image_report)
    #     print(type(image_report))
    #     query = diagnose + image_report["des_dic"] + image_report["con_dic"]
    # else:
    #     query = diagnose    
    
    #召回方案：根据临床诊断召回相应核保结论
    query = diagnose
    
    
    try:
        s_time = time.time()
        #获得相似疾病name
        # map_disease_name= disease_similarity(query,disease_n_2_list)
        # disease_key = map_disease_name[0][0]
        disease_key = row["disease_name"]
        if disease_key in all_dise_conc_keys:
            max_simi_key = disease_key
        else:
            max_simi_key = map_key_2_dise.get(disease_key,"")
        # max_simi_dise_names = map_key_2_dise.get(map_disease_name[0][0],[map_disease_name[0]])
        #获得相似疾病的结论
        # recall_kbs = df_dise_2_conc[df_dise_2_conc["disease_name"]==max_simi_key]["conclusion"].tolist() 
        recall_kbs = df_dise_2_conc[df_dise_2_conc["disease_name"]==disease_key]["conclusion"].tolist()
        time_use = time.time() - s_time
        print("recall time use:",time_use)
        recall_time_use.append(time_use)
        ans_list = recall_kbs
        comments = map_disease_comment.get(disease_key,[])

    except Exception as es:
        print(es)
        ans_list = []

    #构建prompt
    output_struct = {"重疾险": "延期", "防癌": "延期", "护理": "肾功能异常延期", "医疗险": "延期", "意外险": "肾功能异常延期"}
    prompt = """
            你是一个保险公司的专业核保老师，根据提供的病人基本信息和给定的核保结论返回最接近的核保结论。
            病人基本信息:{basic_info}
            核保结论:{ans_list}
            核保结论相关注解:{comments}
            注意：
                1、核保结论的返回格式为:{output_struct}
                2、不要有其他信息说明
                3、诺核保结论为空则不返回最终核保结论
            """
    input_prompt = prompt.format(basic_info=basic_info,ans_list=ans_list,comments=comments,output_struct=output_struct)
    print("input_prompt:",input_prompt)


    #大模型结论生成
    input = [{"role": "user", "content": input_prompt}]
    result = qa_base(input)
    print("label:",label)
    print("result:",result)
    print("**********==**************\n")
    results_underwriting.append(result)
    recall_query.append(ans_list)
    recall_keys.append(disease_key)

    

病人基本信息 年龄:29,性别:女性,临床诊断:子宫肌瘤可能,影像报告:{"report_name": "", "des_dic": "【子宫】  【经阴道】子宫位置：  前位：  子宫大小：  长径  60mm,  左右径  65mm,  前后径  59mm;子宫形态：  不规则：  子宫回声：  不均匀：  肌层彩色血流星点状,内膜厚度12mm    宫颈长度  38mm子宫后壁突起中低回声区：  54*50*50mm", "con_dic": "子宫肌瘤可能。"}
recall time use: 0.0014734268188476562
input_prompt: 
            你是一个保险公司的专业核保老师，根据提供的病人基本信息和给定的核保结论返回最接近的核保结论。
            病人基本信息:年龄:29,性别:女性,临床诊断:子宫肌瘤可能,影像报告:{"report_name": "", "des_dic": "【子宫】  【经阴道】子宫位置：  前位：  子宫大小：  长径  60mm,  左右径  65mm,  前后径  59mm;子宫形态：  不规则：  子宫回声：  不均匀：  肌层彩色血流星点状,内膜厚度12mm    宫颈长度  38mm子宫后壁突起中低回声区：  54*50*50mm", "con_dic": "子宫肌瘤可能。"}
            核保结论:["{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '病历、妇科超声、血常规', '检查结果': '已手术，病理良性', '重疾': '标体', '防癌': '标体', '意外险': '标体', '护理险': '标体', '医疗险': '标体'}", "{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '病历、妇科超声、血常规', '检查结果': '已手术，病理非良性', '重疾': '拒保', '防癌': '拒保', '意外险': '拒保', '护理险': '拒保', '医疗险': '拒保'}", "{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '病历、妇科超声、血常规', '检查结果': '未手术，无手术指征*2', '重疾'

In [52]:
sum(recall_time_use)/len(recall_time_use)

0.32474566996097565

In [22]:
df.columns

Index(['姓名', 'disease_name', '编号（身份证号）', '性别\n（1:男，2:女，0:未知）', '医院', '日期',
       '临床诊断（化验项、疾病等以下划线拼接）', '账单金额（数字）', '年龄', '编号（案件编号）', '图片名/文件名',
       'attach id', '图片分类/票据类别', '影像报告内容image_report',
       'data_source：类别 如昆仑体检告知', '线上页面与结果', 'RAG核保结论', 'recall_disease',
       'recall_conclusions'],
      dtype='object')

In [27]:
df_new = df.drop(['RAG核保结论',
       'recall_disease', 'recall_conclusions'],axis=1)

In [28]:
df_new["RAG核保结论"] = results_underwriting
df_new["recall_disease"] = recall_keys
df_new["recall_conclusions"] = recall_query
df_new.to_csv("./results_edit_distance/核保结论_规则引擎_vs_RAG_editDistanceSearch_20241204.csv",index=False)

In [37]:
len_get_recall = len([rec_res for rec_res in recall_query if len(rec_res)!=0])
print(len_get_recall)
print(f"召回率：{round(len_get_recall/len(recall_query)*100,2)}%")

32
召回率：100.0%


In [38]:
# df_full = pd.read_csv("./核保结论_规则引擎_vs_RAG_FullTextSearch.csv")
# df_full = pd.read_csv("./核保结论_规则引擎_vs_RAG_SemanticSearch.csv")
# df_full = pd.read_csv("./核保结论_规则引擎_vs_RAG_HybridSearch.csv")
# df_full = pd.read_csv("./results_edit_distance/核保结论_规则引擎_vs_RAG_editDistanceSearch.csv")


# recall_query = df_full["recall_query"].tolist()
# len_get_recall = len([rec_res for rec_res in recall_query if len(eval(rec_res))!=0])
# print(len_get_recall)
# print(f"召回率：{round(len_get_recall/len(recall_query)*100,2)}%")
bad_dise = []
for idx,row in df.iterrows():
    if len(row["recall_query"]) == 0:
        bad_dise.append(row["临床诊断（化验项、疾病等以下划线拼接）"])
print(len(bad_dise))
print(bad_dise)

0
[]


In [25]:
df_dise_2_conc[df_dise_2_conc["file_name"]=="子宫.txt"].head(30)

,disease_name,file_name,conclusion
812,子宫出血,子宫.txt,"{'疾病': '子宫出血', '': '功能失调性子宫出血', '资料': '病历、妇科超声..."
813,子宫出血,子宫.txt,"{'疾病': '子宫出血', '': '功能失调性子宫出血', '资料': '病历、妇科超声..."
814,子宫出血,子宫.txt,"{'疾病': '子宫出血', '': '功能失调性子宫出血', '资料': '病历、妇科超声..."
815,子宫出血,子宫.txt,"{'疾病': '子宫出血', '': '非功能失调性子宫出血', '资料': '病历、妇科超..."
816,子宫出血,子宫.txt,"{'疾病': '子宫出血', '': '绝经后子宫出血', '资料': '病历、妇科超声、血..."
817,子宫出血,子宫.txt,"{'疾病': '子宫出血', '': '待进行子宫切除', '资料': '病历、妇科超声、血..."
818,子宫肌瘤,子宫.txt,"{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '..."
819,子宫肌瘤,子宫.txt,"{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '..."
820,子宫肌瘤,子宫.txt,"{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '..."
821,子宫肌瘤,子宫.txt,"{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '..."


In [33]:
query_dise = bad_dise[1]
print(query_dise)
results_query = disease_similarity(query_dise,disease_n_2_list)
print(results_query)
print(results_query[0][0])
map_disease_name = map_key_2_dise.get(results_query[0][0],[results_query[0]])
print(map_disease_name)
df_dise_2_conc[df_dise_2_conc["disease_name"]==map_disease_name].head()

子宫肌瘤可能
[('子宫肌瘤', 0.6666666666666667), ('子宫肉瘤', 0.5), ('子宫切除手术', 0.33333333333333337)]
子宫肌瘤
子宫肌瘤


,disease_name,file_name,conclusion
818,子宫肌瘤,子宫.txt,"{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '..."
819,子宫肌瘤,子宫.txt,"{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '..."
820,子宫肌瘤,子宫.txt,"{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '..."
821,子宫肌瘤,子宫.txt,"{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '..."


In [62]:
#评测
"""
四个召回错误
2个召回正确，生成结论错误
"""

'\n四个召回错误\n2个召回正确，生成结论错误\n'